# 08 — Feature Expansion Experiment

## Purpose

Test whether adding a new economic indicator improves the tuned
XGBoost model on the held-out test period.

## Candidate indicators and coverage analysis

We investigated five additional World Bank indicators for potential
inclusion. Coverage analysis (missing values across the 27 × 31 panel):

| Indicator | Missing | Verdict |
|---|---|---|
| BN.CAB.XOKA.GD.ZS (current account balance) | 5.1% | ✅ tested |
| FS.AST.PRVT.GD.ZS (domestic credit to private sector) | 22.9% | ❌ rejected — structural, pre-2001 |
| TT.PRI.MRCH.XD.WD (terms of trade) | 35.5% | ❌ rejected — all pre-2005 |
| FM.LBL.BMNY.GD.ZS (broad money) | 74.8% | ❌ rejected |
| NY.GDP.PCAP.KD.ZG (GDP per capita growth) | 0% | ⏭️ skipped — redundant with existing features |

**Only the current account balance was viable for testing.**

## Method

We compare the tuned XGBoost (from `05_hyperparameter_tuning.ipynb`)
across three feature sets on the same chronological split:

- **A. Baseline** — 11 original features
- **B. + Current Account** — 11 + `current_account_percent_gdp`
- **C. + Current Account & Lag** — 11 + current account + current account lag

**Note on the 36 imputed rows.** The extra-indicators pivot drops country-years
where the current account is missing (794 rows instead of 837). The subsequent
merge re-introduces those country-years as NaNs, producing 36 missing values
in the model dataset. All 36 fall in the training split (test is fully covered).
They are imputed with country-level medians computed from training rows only.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from src import config, split, preprocessing_extra as ppe

In [3]:
model_df = pd.read_csv(config.MODEL_DATA_PATH)
df = ppe.merge_extra_features(model_df)

# Sort by year for TimeSeriesSplit consistency
df = df.sort_values([config.YEAR_COL, config.COUNTRY_CODE_COL]).reset_index(drop=True)

print("Shape:", df.shape)
print("All columns:", df.columns.tolist())
print("Split:", df[config.SPLIT_COL].value_counts().to_dict())

Shape: (810, 21)
All columns: ['Country Code', 'Country Name', 'Year', 'fdi_inflows_percent_gdp', 'inflation', 'government_consumption_percent_gdp', 'exports_percent_gdp', 'gross_fixed_capital_formation', 'imports_percent_gdp', 'gdp_growth', 'gdp_per_capita', 'unemployment', 'population_growth', 'trade_openness', 'gdp_growth_lag_1', 'gdp_growth_lag_2', 'target_gdp_growth_next_year', 'gdp_per_capita_group', 'dataset_split', 'current_account_percent_gdp', 'current_account_lag_1']
Split: {'train': 675, 'test': 135}


In [4]:
FINAL_PARAMS = dict(
    colsample_bytree=0.7,
    learning_rate=0.035,
    max_depth=6,
    n_estimators=90,
    reg_lambda=1.0,
    subsample=0.85,
)

def make_final_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   XGBRegressor(
            **FINAL_PARAMS,
            random_state=config.RANDOM_STATE,
            n_jobs=-1, verbosity=0,
        )),
    ])

## Experiment

Each feature set is trained with identical hyperparameters, on the same
train/test split. Metrics reported: train MAE, test MAE, train-test MAE
gap (overfitting indicator), train R², test R².

**Decision rule:** if either B or C improves test MAE by more than ~0.1
over A, the new feature is worth integrating. Otherwise it is reported
as a tested-but-rejected feature.

In [5]:
BASE_FEATURES = split.FEATURE_COLS  # your existing 11
EXTRA = ["current_account_percent_gdp"]
EXTRA_LAG = ["current_account_lag_1"]

feature_sets = {
    "A. Baseline (11 features)":         BASE_FEATURES,
    "B. + Current Account":              BASE_FEATURES + EXTRA,
    "C. + Current Account & Lag":        BASE_FEATURES + EXTRA + EXTRA_LAG,
}

train = df[df[config.SPLIT_COL] == "train"]
test  = df[df[config.SPLIT_COL] == "test"]
y_train = train[config.TARGET_COL]
y_test  = test[config.TARGET_COL]

rows = []
for name, feats in feature_sets.items():
    Xtr = train[feats]
    Xte = test[feats]
    m = make_final_model()
    m.fit(Xtr, y_train)
    pred_tr = m.predict(Xtr)
    pred_te = m.predict(Xte)
    rows.append({
        "feature_set":  name,
        "n_features":   len(feats),
        "train_MAE":    mean_absolute_error(y_train, pred_tr),
        "test_MAE":     mean_absolute_error(y_test,  pred_te),
        "MAE_gap":      mean_absolute_error(y_test, pred_te) - mean_absolute_error(y_train, pred_tr),
        "train_R2":     r2_score(y_train, pred_tr),
        "test_R2":      r2_score(y_test,  pred_te),
    })

results = pd.DataFrame(rows).set_index("feature_set").round(3)
print(results)
results.to_csv(config.PROCESSED_DIR / "feature_expansion_results.csv")

                            n_features  train_MAE  test_MAE  MAE_gap  \
feature_set                                                            
A. Baseline (11 features)           11      1.148     2.841    1.693   
B. + Current Account                12      1.123     2.818    1.695   
C. + Current Account & Lag          13      1.154     2.749    1.595   

                            train_R2  test_R2  
feature_set                                    
A. Baseline (11 features)      0.818   -0.332  
B. + Current Account           0.826   -0.279  
C. + Current Account & Lag     0.811   -0.318  


## Result

| Feature set | n | Train MAE | Test MAE | MAE gap | Test R² |
|---|---|---|---|---|---|
| A. Baseline (11 features) | 11 | 1.148 | 2.841 | 1.693 | −0.332 |
| B. + Current Account | 12 | 1.123 | 2.818 | 1.695 | **−0.279** |
| C. + Current Account & Lag | 13 | 1.154 | **2.749** | **1.595** | −0.318 |

**Observations:**

1. **The current account balance produces a small but consistent
   improvement on test MAE.**
   - B vs A: −0.023 MAE (0.8% improvement)
   - C vs A: −0.092 MAE (3.2% improvement)

2. **Test R² improves most in B** (−0.279 vs −0.332 baseline), indicating
   the model explains slightly more variance when the current account is
   added without its lag.

3. **Overfitting does not worsen.** The MAE gap stays essentially flat for
   A and B (1.693 vs 1.695), and *decreases* for C (1.595). The extra
   feature does not come at the cost of generalization.

4. **No configuration clears the decision threshold of ~0.1 MAE.** C
   comes close (−0.092) but does not cross it. B is a smaller improvement.

## Interpretation

The current account balance carries **weak but non-zero signal** for
next-year GDP growth. This is economically plausible: external imbalances
are a slow-moving determinant of growth adjustments, and the empirical
growth literature has linked persistent current account deficits to
eventual slowdowns.

However, the improvement is modest. On a target with standard deviation
~3.4 percentage points, a 0.02–0.09 MAE reduction is small enough that it
could partly reflect noise. We therefore treat this as a **tested but
not integrated** feature: the current account is documented here for
completeness, but the final model in `06_evaluation_and_findings.ipynb`
retains the original 11-feature set.

## Conclusion

**Adding the current account balance and its lag does not change the
project's central finding.** The tuned XGBoost still does not beat the
naive mean baseline (test MAE 2.49). The feature expansion experiment
confirms that the difficulty of this prediction problem is not primarily
due to insufficient feature coverage — at least not from the additional
indicators tested here.

Four of five candidate indicators were rejected on data-availability
grounds; the single viable one provides a small, non-decisive
improvement. This is a legitimate negative result and supports the
project's overall narrative: **next-year GDP growth across the EU-27 is
not reliably predictable from World Bank macro indicators, regardless of
feature engineering.**